# Vector Search for Retrieval — companion notebook

Sweep the score threshold and watch the trade you are really choosing: **false abstentions** against **false answers**.

Standard library plus matplotlib. No embedding model needed.

## 1. A tiny vector space

Four documents, hand-made 4-D vectors so the ranking can be checked by eye. Swap in real embeddings and only the numbers change.

In [1]:
import math

DOCS = {
    "refund policy":   [0.94, 0.20, 0.10, 0.02],
    "shipping times":  [0.18, 0.95, 0.12, 0.03],
    "opening hours":   [0.09, 0.14, 0.96, 0.05],
    "office location": [0.11, 0.22, 0.44, 0.07],
}

def dot(a, b): return sum(x*y for x, y in zip(a, b))
def norm(a):   return math.sqrt(dot(a, a))
def cosine(a, b): return dot(a, b) / (norm(a) * norm(b))

def top_k(qv, k=2):
    return sorted(((cosine(qv, v), n) for n, v in DOCS.items()), reverse=True)[:k]

for name, v in DOCS.items():
    print(f"{name:<18} {v}")

refund policy      [0.94, 0.2, 0.1, 0.02]
shipping times     [0.18, 0.95, 0.12, 0.03]
opening hours      [0.09, 0.14, 0.96, 0.05]
office location    [0.11, 0.22, 0.44, 0.07]


## 2. Answerable vs unanswerable queries

The key asymmetry: some questions have a real answer in the corpus and some do not. Top-k cannot tell them apart — it returns *k* results either way.

In [2]:
ANSWERABLE = {
    "how do I get a refund": [0.90, 0.25, 0.08, 0.01],
    "when do you close":     [0.10, 0.12, 0.93, 0.04],
    "where are you based":   [0.12, 0.20, 0.40, 0.06],
    "shipping how long":     [0.20, 0.92, 0.14, 0.02],
}
OUT_OF_SCOPE = {
    "what is the weather":   [0.03, 0.05, 0.06, 0.97],
    "tell me a joke":        [0.06, 0.04, 0.09, 0.88],
    "who won the football":  [0.05, 0.08, 0.05, 0.93],
}

print(f"{'query':<24} {'best score':>10}  best match")
for label, group in (("ANSWERABLE", ANSWERABLE), ("OUT OF SCOPE", OUT_OF_SCOPE)):
    print(f"\n-- {label} --")
    for q, qv in group.items():
        s, n = top_k(qv, 1)[0]
        print(f"  {q:<22} {s:>10.3f}  {n}")

query                    best score  best match

-- ANSWERABLE --
  how do I get a refund       0.998  refund policy
  when do you close           1.000  opening hours
  where are you based         0.999  office location
  shipping how long           0.999  shipping times

-- OUT OF SCOPE --
  what is the weather         0.219  office location
  tell me a joke              0.258  office location
  who won the football        0.231  office location


## 3. The frontier you are actually choosing

Sweep the threshold. Every value trades **false abstentions** (refusing an answerable question) against **false answers** (answering an unanswerable one). There is generally no value that avoids both — the distributions overlap.

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

grid = [i / 100 for i in range(0, 101)]
fa, fab = [], []          # false answers, false abstentions
for t in grid:
    fa.append(sum(1 for qv in OUT_OF_SCOPE.values() if top_k(qv, 1)[0][0] >= t))
    fab.append(sum(1 for qv in ANSWERABLE.values() if top_k(qv, 1)[0][0] < t))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, fa, label="false ANSWERS (answered an out-of-scope query)")
ax.plot(grid, fab, label="false ABSTENTIONS (refused an answerable query)")
ax.set_xlabel("score threshold")
ax.set_ylabel("number of queries")
ax.set_title("Every threshold is a trade")
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

safe = [t for t, a, b in zip(grid, fa, fab) if a == 0 and b == 0]
print(f"thresholds with ZERO of both: {len(safe)} "
      f"({f'{min(safe):.2f} to {max(safe):.2f}' if safe else 'none'})")
print("On real data the two distributions overlap and this window is usually EMPTY.")

thresholds with ZERO of both: 74 (0.26 to 0.99)
On real data the two distributions overlap and this window is usually EMPTY.


## 4. What overlap does to you

The clean separation above is an artefact of a toy corpus: every out-of-scope query is about a *completely* different topic, so it scores low. Real unanswerable questions are often **topically identical** to something you do have — and score high. Add one and watch the safe window collapse.

In [4]:
# Unanswerable, but topically identical to a document we DO have.
# The corpus lists regular opening hours; it says nothing about holidays.
OUT_OF_SCOPE["what are your holiday hours"] = [0.10, 0.15, 0.94, 0.05]

s, n = top_k(OUT_OF_SCOPE["what are your holiday hours"], 1)[0]
print(f"unanswerable query scores {s:.3f} against {n!r}")
print("-- higher than several ANSWERABLE queries. No threshold separates it.\n")

fa2, fab2 = [], []
for t in grid:
    fa2.append(sum(1 for qv in OUT_OF_SCOPE.values() if top_k(qv, 1)[0][0] >= t))
    fab2.append(sum(1 for qv in ANSWERABLE.values() if top_k(qv, 1)[0][0] < t))
safe2 = [t for t, a, b in zip(grid, fa2, fab2) if a == 0 and b == 0]
print(f"thresholds with zero of both: {len(safe2)}   (was {len(safe)})\n")

print(f"{'threshold':>10} {'false answers':>14} {'false abstentions':>18}")
for t in (0.3, 0.5, 0.7, 0.9, 0.95):
    a = sum(1 for qv in OUT_OF_SCOPE.values() if top_k(qv, 1)[0][0] >= t)
    b = sum(1 for qv in ANSWERABLE.values() if top_k(qv, 1)[0][0] < t)
    print(f"{t:>10.2f} {a:>14} {b:>18}")
print("\nA threshold catches IRRELEVANT retrieval. It cannot catch retrieval that is")
print("relevant and still unable to answer. That needs a different mechanism.")

unanswerable query scores 1.000 against 'opening hours'
-- higher than several ANSWERABLE queries. No threshold separates it.

thresholds with zero of both: 0   (was 74)

 threshold  false answers  false abstentions
      0.30              1                  0
      0.50              1                  0
      0.70              1                  0
      0.90              1                  0
      0.95              1                  0

A threshold catches IRRELEVANT retrieval. It cannot catch retrieval that is
relevant and still unable to answer. That needs a different mechanism.


## 5. Your turn

1. Replace the vectors with real embeddings of your own documents and queries.
2. Assemble ~30 answerable and ~30 out-of-scope questions and redraw §3.
3. Decide which error is worse for your product, then choose the threshold to match — a wrong answer and a needless refusal rarely cost the same.